# Credit Risk: Hazard Rates

This notebook introduces the **hazard rate** (default intensity) — the fundamental building block of credit risk modelling.

We cover:
1. The default event as a random variable and the survival probability
2. The hazard rate: definition and intuition
3. Computing survival probabilities: a numerical example
4. The constant hazard rate: exponential default times
5. The credit triangle: linking hazard rate, spread, and recovery
6. The piecewise-constant hazard rate model used by the `credit` package
7. Shape of the hazard rate term structure
8. Bootstrapping: extracting hazard rates from market CDS spreads

For CDS pricing and bootstrapping, see **`03_cds_pricing.ipynb`**.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
sys.path.insert(0, str(root))

from database import set_db_path
set_db_path(str(root / 'examples' / 'demo.db'))

In [ ]:
import sqlite3
from database import get_db_path
from scripts.initialise import init_db, _seed_holidays

init_db()
with sqlite3.connect(get_db_path()) as conn:
    count = conn.execute("SELECT COUNT(*) FROM holidays").fetchone()[0]
    if count == 0:
        _seed_holidays(conn)
        print("Seeded demo.db with holiday data.")
    else:
        print(f"demo.db already seeded ({count} holidays). Skipping.")

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import date
from dateutil.relativedelta import relativedelta

from credit import SurvivalCurve
from market_conventions import DayCountConvention

REF = date(2026, 3, 27)
ACT365 = DayCountConvention.ACT_365_FIXED

## 1. The Default Event

Let $\tau$ be the (random) time at which a reference entity defaults. We never observe $\tau$ directly in advance — we only know at any future date $t$ whether $\tau > t$ (still alive) or $\tau \le t$ (defaulted).

The central object is the **survival probability**:

$$Q(t) = P(\tau > t)$$

$Q(t)$ is a decreasing function from $Q(0) = 1$ toward zero. It encodes everything we know about the timing of default.

Closely related is the **default probability** up to time $t$:

$$PD(t) = 1 - Q(t) = P(\tau \le t)$$

The **marginal default probability** — the probability of defaulting in the small interval $[t, t + dt]$, regardless of the path — is:

$$P(\tau \in [t, t+dt]) = -dQ(t) = Q(t) - Q(t+dt)$$

This is not directly useful for pricing, because it mixes entities that are *already* alive at $t$ with those that defaulted before $t$. For pricing, we need the **conditional** view.

## 2. The Hazard Rate (Default Intensity)

The **hazard rate** $\lambda(t)$ is defined as the instantaneous conditional default probability, given survival to $t$:

$$P(\tau \in [t, t+dt] \mid \tau > t) = \lambda(t) \, dt$$

Intuitively: if you are alive at $t$, what is the probability you default in the next instant? $\lambda(t)$ answers that question, independently of history prior to $t$.

### Deriving the link to $Q(t)$

Expand the left-hand side using Bayes' rule and the definition of $Q$:

$$P(\tau \in [t, t+dt] \mid \tau > t) = \frac{P(\tau \in [t, t+dt])}{P(\tau > t)} = \frac{Q(t) - Q(t+dt)}{Q(t)}$$

Setting this equal to $\lambda(t)\,dt$ and dividing both sides by $dt$:

$$\lambda(t) = \frac{Q(t) - Q(t+dt)}{Q(t)\,dt}$$

Taking the limit $dt \to 0$, the numerator is $-dQ/dt$ by definition of the derivative:

$$\lambda(t) = -\frac{Q'(t)}{Q(t)}$$

The right-hand side is exactly the chain rule for $\frac{d}{dt}\ln Q(t) = \frac{Q'(t)}{Q(t)}$, so:

$$\boxed{\lambda(t) = -\frac{d}{dt} \ln Q(t)}$$

The $\ln$ is not an assumption — it is a consequence of the division by $Q(t)$ that arises from conditioning on survival.

### Inverting to get $Q(t)$ from $\lambda$

Integrating both sides from $0$ to $t$:

$$\int_0^t \lambda(s)\,ds = -\ln Q(t) + \underbrace{\ln Q(0)}_{=\,0}$$

Exponentiating:

$$Q(t) = \exp\!\left(-\int_0^t \lambda(s)\,ds\right)$$

The integral $H(t) = \int_0^t \lambda(s)\,ds$ is called the **cumulative hazard**. All information about survival is captured in $H(t)$.

### Why hazard rates and not survival probabilities directly?

- Hazard rates are **local**: each pillar independently describes default intensity over its segment. This makes bootstrapping straightforward — solve one pillar at a time.
- Survival probabilities are **global**: changing one near-term pillar shifts all longer-dated survival probabilities.
- Hazard rates must be $\ge 0$; survival probabilities must be monotonically decreasing. Parameterising via hazard rates enforces both constraints automatically.

## 3. Computing Survival Probabilities — Numerical Example

Given the formula $Q(t) = \exp(-H(t))$ and a piecewise-constant hazard rate, the cumulative hazard at each pillar accumulates as a simple sum:

$$H(t_i) = \sum_{j=1}^{i} \lambda_j \cdot (t_j - t_{j-1})$$

Each segment contributes independently — there is no interaction between segments. This separability is what makes the model easy to bootstrap.

**Example**: three pillars at 1Y, 3Y, 5Y with hazard rates $\lambda_1 = 1.50\%$, $\lambda_2 = 2.00\%$, $\lambda_3 = 3.50\%$:

| Segment | $\lambda$ | $\Delta t$ | $\lambda \cdot \Delta t$ | $H(t)$ | $Q(t) = e^{-H(t)}$ | $PD(t) = 1 - Q(t)$ |
|---|---|---|---|---|---|---|
| $[0, 1\text{Y}]$ | 1.50% | 1 yr | 0.0150 | 0.0150 | $e^{-0.0150} = 98.51\%$ | 1.49% |
| $(1\text{Y}, 3\text{Y}]$ | 2.00% | 2 yr | 0.0400 | 0.0550 | $e^{-0.0550} = 94.65\%$ | 5.35% |
| $(3\text{Y}, 5\text{Y}]$ | 3.50% | 2 yr | 0.0700 | 0.1250 | $e^{-0.1250} = 88.25\%$ | 11.75% |

Note that $PD(t)$ is the **cumulative** probability of having defaulted by time $t$, not the probability of defaulting in a specific period.

In [ ]:
import math

# Three-pillar example matching the table above
segments = [
    ("1Y", 1.0, 0.0150),
    ("3Y", 3.0, 0.0200),
    ("5Y", 5.0, 0.0350),
]

print(f"{'Pillar':<8} {'λ (bps)':>10} {'Δt':>5} {'λ·Δt':>8} {'H(t)':>8} {'Q(t)':>9} {'PD(t)':>9}")
print("-" * 62)

H = 0.0
t_prev = 0.0
for label, t, lam in segments:
    delta_t = t - t_prev
    dH = lam * delta_t
    H += dH
    Q = math.exp(-H)
    print(f"{label:<8} {lam*10_000:>9.0f}bp  {delta_t:>3.0f}Y  {dH:>7.4f}  {H:>7.4f}  {Q:>8.2%}  {1-Q:>8.2%}")
    t_prev = t

# Cross-check against SurvivalCurve
print()
print("Cross-check against SurvivalCurve:")
sc_ex = SurvivalCurve(
    reference_date=REF,
    pillar_dates=[REF + relativedelta(years=1),
                  REF + relativedelta(years=3),
                  REF + relativedelta(years=5)],
    hazard_rates=[0.0150, 0.0200, 0.0350],
    day_count_convention=ACT365,
)
for label, t, _ in segments:
    d = REF + relativedelta(years=int(t))
    q = sc_ex.survival_probability(d)
    print(f"  Q({label}) = {q:.4%}  PD({label}) = {1-q:.4%}")

## 4. Constant Hazard Rate — Exponential Default Times

Suppose the hazard rate is constant: $\lambda(t) = \lambda$ for all $t$. Plugging into the survival probability formula from Section 2:

$$Q(t) = \exp\!\left(-\int_0^t \lambda\,ds\right) = e^{-\lambda t}$$

This is the **exponential distribution** — it is memoryless: the conditional probability of surviving another year is the same regardless of how long the entity has already survived.

### Recovering $\lambda$ from $Q(t)$

Starting from the general inversion formula (Section 2):

$$\lambda(t) = -\frac{d}{dt} \ln Q(t)$$

Substitute $Q(t) = e^{-\lambda t}$:

$$\lambda(t) = -\frac{d}{dt} \ln e^{-\lambda t} = -\frac{d}{dt}(-\lambda t) = \lambda \quad \forall\, t$$

The implied hazard rate is **constant** — the term structure is flat. This is the only functional form of $Q(t)$ for which $\lambda(t)$ is independent of $t$.

Equivalently, $\ln Q(t)$ is **linear** in $t$ with slope $-\lambda$. Any deviation from log-linearity signals a non-flat hazard rate term structure — a fact exploited directly by the piecewise-constant bootstrap in Section 6.

Typical investment-grade names trade at 50–150 bps, implying hazard rates of roughly 1–3%. High-yield names can trade at 300–1000+ bps.

In [ ]:
# Verify numerically: recover λ(t) = -d/dt ln Q(t) from Q(t) = exp(-λt)
# Finite difference: λ(t) ≈ -[ln Q(t+dt) - ln Q(t)] / dt

lam_true = 0.05  # 5% hazard rate
t_grid = np.linspace(0.01, 10, 1000)
dt = t_grid[1] - t_grid[0]

Q = np.exp(-lam_true * t_grid)
lam_implied = -np.diff(np.log(Q)) / dt
t_mid = (t_grid[:-1] + t_grid[1:]) / 2

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: Q(t) and ln Q(t) — log-linearity is the signature of a flat hazard rate
axes[0].plot(t_grid, Q * 100, color="#2166ac", linewidth=2, label="Q(t)")
axes[0].set_xlabel("Years")
axes[0].set_ylabel("Q(t) (%)", color="#2166ac")
axes[0].tick_params(axis="y", colors="#2166ac")
axes[0].set_title(r"$Q(t) = e^{-\lambda t}$, $\lambda = 5\%$")
axes[0].grid(True, alpha=0.3)

ax0b = axes[0].twinx()
ax0b.plot(t_grid, np.log(Q), color="#d6604d", linewidth=1.5, linestyle="--", label=r"$\ln Q(t)$")
ax0b.set_ylabel(r"$\ln Q(t)$  [slope = $-\lambda$]", color="#d6604d")
ax0b.tick_params(axis="y", colors="#d6604d")
ax0b.legend(loc="upper right")

# Right: recovered hazard rate — must be flat
axes[1].plot(t_mid, lam_implied * 100, color="#d6604d", linewidth=2,
             label=r"$\hat{\lambda}(t) = -\frac{d}{dt}\ln Q(t)$")
axes[1].axhline(lam_true * 100, color="black", linewidth=1, linestyle=":",
                label=f"True $\\lambda$ = {lam_true*100:.0f}%")
axes[1].set_xlabel("Years")
axes[1].set_ylabel("Implied hazard rate (%)")
axes[1].set_title("Recovered hazard rate — flat by construction")
axes[1].set_ylim(0, 8)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Max |lambda_implied - lambda_true| = {abs(lam_implied - lam_true).max():.2e}  (finite-difference rounding only)")

In [ ]:
t = np.linspace(0, 10, 500)

hazard_rates = {
    "λ = 0.5% (IG tight)": 0.005,
    "λ = 2%   (IG)": 0.02,
    "λ = 5%   (HY)": 0.05,
    "λ = 15%  (distressed)": 0.15,
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

colors = ["#2166ac", "#4dac26", "#d6604d", "#762a83"]

for (label, lam), color in zip(hazard_rates.items(), colors):
    q = np.exp(-lam * t)
    axes[0].plot(t, q * 100, label=label, color=color, linewidth=2)
    axes[1].axhline(lam * 100, label=label, color=color, linewidth=2)

axes[0].set_xlabel("Years")
axes[0].set_ylabel("Survival probability (%)")
axes[0].set_title("Q(t) = exp(−λt) for constant hazard rates")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 101)

axes[1].set_xlabel("Years")
axes[1].set_ylabel("Hazard rate (%)")
axes[1].set_title("Constant (flat) hazard rate term structure")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 10)
axes[1].set_ylim(-0.5, 18)

plt.tight_layout()
plt.show()

## 5. The Credit Triangle

The simplest relationship between hazard rate, CDS spread, and recovery rate is:

$$s \approx \lambda \cdot (1 - R) \quad \Longleftrightarrow \quad \lambda \approx \frac{s}{1 - R}$$

where $s$ is the par CDS spread and $R$ is the recovery rate.

**Derivation sketch** (continuous limit, flat rates, flat hazard):
- Premium leg PV per unit notional: $\int_0^T s \cdot Q(t) \cdot df(t)\,dt \approx s / (\lambda + r) \cdot (1 - e^{-(\lambda+r)T})$
- Protection leg PV: $(1-R) \int_0^T \lambda Q(t) df(t)\,dt \approx \lambda(1-R)/(\lambda+r) \cdot (1 - e^{-(\lambda+r)T})$
- Setting them equal: $s = \lambda(1-R)$

This is approximate in practice (discrete payments, non-flat rates, accrued-on-default), but useful for intuition and back-of-the-envelope checks.

**Example:** A 5Y CDS trading at 250 bps with 40% recovery implies $\lambda \approx 250 / (1 - 0.40) = 417$ bps ≈ 4.2% per year.

In [ ]:
# Credit triangle: λ ≈ s / (1 - R)
spreads_bps = np.arange(10, 1001, 10)
spreads = spreads_bps / 10_000

recovery_rates = {"R = 20%": 0.20, "R = 40%": 0.40, "R = 60%": 0.60}

fig, ax = plt.subplots(figsize=(9, 4.5))

for (label, R), color in zip(recovery_rates.items(), ["#2166ac", "#4dac26", "#d6604d"]):
    lam = spreads / (1 - R)
    ax.plot(spreads_bps, lam * 10_000, label=label, color=color, linewidth=2)

ax.plot(spreads_bps, spreads_bps, color="black", linewidth=1, linestyle="--", label="R = 0% (λ = s)")

ax.set_xlabel("CDS spread (bps)")
ax.set_ylabel("Implied hazard rate (bps)")
ax.set_title("Credit triangle: implied hazard rate λ ≈ s / (1 − R)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Back-of-envelope implied hazard rates (R = 40%):")
print(f"{'Spread':>10} {'λ (bps)':>12} {'5Y Q':>10}")
print("-" * 36)
for s_bps in [50, 100, 200, 500, 1000]:
    lam = (s_bps / 10_000) / (1 - 0.40)
    q5 = math.exp(-lam * 5)
    print(f"{s_bps:>8} bps  {lam*10_000:>10.0f} bps  {q5:>9.2%}")

## 6. Piecewise-Constant Hazard Rate Model

The market standard for CDS curve construction uses a **piecewise-constant hazard rate** $\lambda(t)$:

$$\lambda(t) = \lambda_i \quad \text{for } t \in (t_{i-1}, t_i]$$

where $0 = t_0 < t_1 < \cdots < t_n$ are the pillar dates (CDS maturities). The cumulative hazard is then:

$$H(t) = \sum_{j=1}^{i-1} \lambda_j (t_j - t_{j-1}) + \lambda_i (t - t_{i-1}), \quad t \in (t_{i-1}, t_i]$$

and the survival probability is $Q(t) = e^{-H(t)}$.

**Key properties:**
- The hazard rate is a step function — flat between pillars, with jumps at pillar dates.
- The survival probability is continuous and piecewise log-linear in time.
- Each pillar's hazard rate is solved independently during bootstrapping (given earlier pillars are already fixed), making the problem separable.
- The model exactly reprices all input CDS quotes by construction.

The `SurvivalCurve` class implements this model directly.

In [ ]:
# Build a SurvivalCurve manually with a realistic upward-sloping hazard rate term structure
pillar_dates = [
    REF + relativedelta(years=1),
    REF + relativedelta(years=2),
    REF + relativedelta(years=3),
    REF + relativedelta(years=5),
    REF + relativedelta(years=7),
    REF + relativedelta(years=10),
]

# Upward-sloping: IG name with modest term premium
hazard_rates_ig = [0.0100, 0.0120, 0.0140, 0.0170, 0.0195, 0.0220]

sc_ig = SurvivalCurve(
    reference_date=REF,
    pillar_dates=pillar_dates,
    hazard_rates=hazard_rates_ig,
    day_count_convention=ACT365,
)

# Monthly survival probability grid
monthly_dates = [REF + relativedelta(months=m) for m in range(1, 121)]
t_axis = [(d - REF).days / 365.25 for d in monthly_dates]
survival_ig = [sc_ig.survival_probability(d) for d in monthly_dates]

pillar_t = [(d - REF).days / 365.25 for d in pillar_dates]
pillar_q = [sc_ig.survival_probability(d) for d in pillar_dates]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: step-function hazard rate
t_step, h_step = [0], [hazard_rates_ig[0] * 100]
for i, (t_p, h) in enumerate(zip(pillar_t, hazard_rates_ig)):
    h_step.append(h * 100)
    t_step.append(t_p)
    if i + 1 < len(pillar_t):
        h_step.append(hazard_rates_ig[i + 1] * 100)
        t_step.append(t_p)
t_step.append(10)
h_step.append(hazard_rates_ig[-1] * 100)

axes[0].plot(t_step, h_step, color="#d6604d", linewidth=2, label="Hazard rate λ(t)")
axes[0].scatter(pillar_t, [h * 100 for h in hazard_rates_ig], color="#d6604d", s=60, zorder=5)
axes[0].set_xlabel("Years")
axes[0].set_ylabel("Hazard rate (%)")
axes[0].set_title("Piecewise-constant hazard rate term structure")
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 10)
axes[0].set_ylim(0, 3.5)

# Right: survival probability
axes[1].plot(t_axis, [q * 100 for q in survival_ig], color="#2166ac", linewidth=2, label="Q(t)")
axes[1].scatter(pillar_t, [q * 100 for q in pillar_q], color="#2166ac", s=60, zorder=5, label="Pillar dates")
axes[1].set_xlabel("Years")
axes[1].set_ylabel("Survival probability (%)")
axes[1].set_title("Implied survival probability")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(75, 101)

plt.tight_layout()
plt.show()

print(f"{'Pillar':<8} {'λ (bps)':>10} {'Q(t)':>10} {'PD(t)':>10}")
print("-" * 42)
labels = ["1Y", "2Y", "3Y", "5Y", "7Y", "10Y"]
for label, d, h, q in zip(labels, pillar_dates, hazard_rates_ig, pillar_q):
    print(f"{label:<8} {h*10_000:>9.0f}bp  {q:>9.2%}  {(1-q):>9.2%}")

## 7. Shape of the Hazard Rate Term Structure

Just like interest rate curves, hazard rate curves come in different shapes depending on market sentiment:

| Shape | Implication |
|---|---|
| **Upward-sloping** | Market prices higher default intensity over time — typical for investment-grade names |
| **Flat** | Constant default intensity across maturities — a simplifying assumption |
| **Inverted (downward-sloping)** | Near-term default risk elevated — typical for distressed or stressed names |

An inverted CDS spread curve is a strong distress signal: the market demands a high premium for near-term protection but expects either default or recovery (and thus lower long-term risk) over the medium term.

In [ ]:
# Three hazard rate term structures
curves = {
    "Upward-sloping (IG)": [0.0100, 0.0120, 0.0140, 0.0170, 0.0195, 0.0220],
    "Flat": [0.0150] * 6,
    "Inverted (distressed)": [0.0600, 0.0500, 0.0400, 0.0280, 0.0200, 0.0150],
}
curve_colors = ["#2166ac", "#4dac26", "#d6604d"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for (name, hrs), color in zip(curves.items(), curve_colors):
    sc = SurvivalCurve(
        reference_date=REF,
        pillar_dates=pillar_dates,
        hazard_rates=hrs,
        day_count_convention=ACT365,
    )
    qs = [sc.survival_probability(d) for d in monthly_dates]

    # Build step function for hazard rate plot
    t_step, h_step = [0], [hrs[0] * 100]
    for i, (t_p, h) in enumerate(zip(pillar_t, hrs)):
        h_step.append(h * 100)
        t_step.append(t_p)
        if i + 1 < len(pillar_t):
            h_step.append(hrs[i + 1] * 100)
            t_step.append(t_p)
    t_step.append(10)
    h_step.append(hrs[-1] * 100)

    axes[0].plot(t_step, h_step, color=color, linewidth=2, label=name)
    axes[1].plot(t_axis, [q * 100 for q in qs], color=color, linewidth=2, label=name)

for ax in axes:
    ax.set_xlabel("Years")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

axes[0].set_ylabel("Hazard rate (%)")
axes[0].set_title("Hazard rate term structure — three shapes")
axes[0].set_xlim(0, 10)

axes[1].set_ylabel("Survival probability (%)")
axes[1].set_title("Corresponding survival curves")

plt.tight_layout()
plt.show()

## 8. Bootstrapping: Extracting Hazard Rates from Market CDS Spreads

In practice we do not observe hazard rates directly. The market quotes **par CDS spreads** — the coupon that makes a CDS worth zero at inception. We must infer hazard rates from these spreads.

The bootstrap proceeds pillar by pillar:

1. Fix all previously solved hazard rates $\lambda_1, \ldots, \lambda_{i-1}$.
2. Solve for $\lambda_i$ such that the implied par spread for the $i$-th CDS matches the market quote.
3. Because the problem is separable (earlier pillars are frozen), each step is a one-dimensional root-finding problem — solved here via bisection.

The par spread formula (per unit notional, discretised over the accrual schedule) is:

$$s^* = \frac{(1-R) \sum_i \frac{df_s + df_e}{2} (Q_{s_i} - Q_{e_i})}{\sum_i \left[ \delta_i \cdot df_{\text{pay},i} \cdot Q_{e_i} + \frac{\delta_i}{2} \cdot \frac{df_s + df_e}{2} (Q_{s_i} - Q_{e_i}) \right]}$$

where $Q_{s_i}, Q_{e_i}$ are survival probabilities at accrual start/end, $df$ are discount factors, $\delta_i$ is the day count fraction, and $R$ is recovery.

The second term in the denominator is the **accrued-on-default** correction: if default occurs mid-period, the protection buyer owes a partial coupon. The midpoint $\frac{df_s + df_e}{2}$ approximates the discount factor at the default time.

For a full worked example with code, see **`03_cds_pricing.ipynb`**.

In [ ]:
# Credit triangle vs exact bootstrap: how good is the approximation?
# We compare λ ≈ s/(1-R) against the actual bootstrapped hazard rates from Section 5.

# Approximate implied spreads from the IG hazard rates via credit triangle: s ≈ λ(1-R)
RECOVERY = 0.40
approx_spreads_bps = [h * (1 - RECOVERY) * 10_000 for h in hazard_rates_ig]

print("Credit triangle approximation quality (R = 40%, upward-sloping curve):")
print(f"{'Pillar':<8} {'λ (bps)':>10} {'λ(1-R) [bps]':>15} {'Approx spread':>16}")
print("-" * 54)
for label, h, s_approx in zip(labels, hazard_rates_ig, approx_spreads_bps):
    print(f"{label:<8} {h*10_000:>9.0f}bp  {s_approx:>14.1f}bp  (≈ {h*(1-RECOVERY)*10_000:.1f} bps spread)")

print()
print("The credit triangle is exact in the continuous-payment, flat-rates limit.")
print("In practice it overstates the hazard rate by ~5% for long maturities")
print("because the discrete accrual schedule and non-flat discount curve matter.")

## Summary

| Concept | Definition | Key formula |
|---|---|---|
| Survival probability | $Q(t) = P(\tau > t)$ | Decreasing from 1 to 0 |
| Hazard rate | Conditional instantaneous default probability | $P(\tau \in [t,t+dt]\mid\tau>t) = \lambda(t)\,dt$ |
| Survival ↔ hazard | Exponential link | $Q(t) = \exp\bigl(-\int_0^t\lambda(s)\,ds\bigr)$ |
| Piecewise-constant | Model used in practice | $\lambda(t) = \lambda_i$ on each segment |
| Credit triangle | Back-of-envelope approximation | $\lambda \approx s/(1-R)$ |
| Bootstrap | Extract hazard rates from market spreads | Sequential bisection, pillar by pillar |

**Next:** See `03_cds_pricing.ipynb` for the full bootstrapping workflow, CDS pricing, and Greeks (CS01, RR01).